# Intro

In this exercise we use a weaviate container that is running to connect to. Checkout the notebook in the root directory in case the container is not running and you need to start it up first.

In [ ]:
!docker ps

In [ ]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter
from typing import List
from tqdm import tqdm
import joblib
import weaviate
import re
from weaviate.util import generate_uuid5
from pprint import pprint
import os
from dotenv import load_dotenv
load_dotenv()

## Connect to weaviate

https://academy.weaviate.io/courses/wa101t-py/m3/p3

- using context manager
- or without

In [ ]:
# check if weaviate is running on the local port
!curl -i http://localhost:8080/v1/.well-known/ready

In [ ]:
import weaviate

with weaviate.connect_to_local(skip_init_checks=True) as client:
    print("Ready:", client.is_ready())
    # do your stuff
# automatically closes connection here


In [ ]:
import weaviate
import os

headers = {
    "X-Openai-Api-Key": os.getenv("OPENAI_API_KEY")
}  # Replace with your Cohere API key

client = weaviate.connect_to_local(headers=headers)

# check if the client is ready
assert client.is_ready()

<a id='2'></a>
## 2 - Configuring the database

---

In this section, you will explore the central object in this lab and in this assignment: [the collection](https://weaviate.io/developers/weaviate/manage-data/collections) - this is the name Weaviate gives to a group of data objects which will be indexed for retrieval. Remember the workflow from the lectures:

<div align="center">
  <img src="./images/workflow.png" alt="RAG Overview" width="60%">
</div>


<a id='2-1'></a>
### 2.1 Creating a Collection

To create a collection, there are some parameters that must be set. The most important for our purposes are:

- `name`: the collection name, this is the name that will be saved in memory and the name that you will need to load it.
- `vectorizer_config`: a list with vectorizer configurations. You can pass more than one vectorizer configuration, which means that in the same vector database, you can vectorize your datapoints with different embedding models. In your context, you will be using only one.

Let's load a database to illustrate this section.

In [ ]:
data = joblib.load("data.joblib")
print(len(data))
data[2]

The dataset is a set of places to visit, with some properties describing each location. The properties here are `place, state, description, best_season_to_visit, attractions, budget, user_ratings, last_updated`. When creating a collection, you must create one property for each key in this dictionary and add the expected datatype. 

<a id='2-2'></a>
### 2.2 Configuring the Vectorizer

As mentioned before, you will use the `text2vec_transformers` embedding model to vectorize your data. To configure it, you must pass the corresponding Configure object. When configuring the vectorizer, you can pass a list of different vectorizers, so your collection can store several vectorizations for the same object. You can also choose to vectorize specific properties on specific vectorizers. In this course you will stick with one vectorizer. Not every property must be vectorized, it depends on the data and the information you want to retrieve.

In this case, let's use the following properties to be vectorized:

`place, state, description, best_season_to_visit, attractions, budget`

These properties will be appended to each other and then vectorized. When defining the property, you might choose to add the property name or not in the vectorization. Note that it would make sense to have the property name in budget, for example, as only the word "Moderate" would not provide enough information about what "Moderate" stands for.

In [ ]:
vectorizer_config = [Configure.NamedVectors.text2vec_openai(
                name="vector", # This is the name you will need to access the vectors of the objects in your collection
                source_properties=['place', 'state', 'description', 'best_season_to_visit', 'attractions', 'budget'], # which properties should be used to generate a vector, they will be appended to each other when vectorizing
                vectorize_collection_name = False, # This tells the client to not vectorize the collection name. 
                                                   # If True, it will be appended at the beginning of the text to be vectorized
                # inference_url="http://127.0.0.1:5000", # Since we are using an API based vectorizer, you need to pass the URL used to make the calls 
                                                       # This was setup in our Flask application
            )]

<a id='2-3'></a>
### 2.3 The Properties

In a collection, the features of each data point are called Properties.

In [ ]:
# Delete the collection in case it exists
if client.collections.exists("example_collectiom"):
    client.collections.delete("example_collection")
    

In [ ]:
if not client.collections.exists('example_collection'): # Creates only if the collection does not exist
    collection = client.collections.create(
            name='example_collection',
            vectorizer_config=vectorizer_config, # The config we defined before,
            #reranker_config=Configure.Reranker.transformers(), # The reranker config

            properties=[  # Define properties
            Property(name="place",vectorize_property_name=True,data_type= DataType.TEXT),
            Property(name="state",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="description",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="best_season_to_visit",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="attractions",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="budget",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="user_ratings", data_type=DataType.NUMBER),
            Property(name="last_updated", data_type=DataType.DATE),

        ]
        )
else:
    collection = client.collections.get("example_collection")

Running it creates a collection and returns the collection. Printing it shows the collection configuration.

In [ ]:
print(collection)

If you try to create a collection that already exists, an exception will be thrown:

In [ ]:
try:
    collection = client.collections.create(
        name='example_collection',

        vectorizer_config=vectorizer_config, # The config we defined before,
    
        properties=[  # Define properties
        Property(name="place",vectorize_property_name=True,data_type= DataType.TEXT),
        Property(name="state",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="description",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="best_season_to_visit",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="attractions",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="budget",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="user_ratings", data_type=DataType.NUMBER),
        Property(name="last_updated", data_type=DataType.DATE),
                 
    ]
    )
except Exception as e:
    print(e)

You can also retrieve all the collections saved:

In [ ]:
client.collections.list_all().keys()

The result of .list_all() is a dictionary with the collections names as keys and their properties.

<a id='2-4'></a>
### 2.4 Adding elements into a Collection

Once you create a collection, you get an empty collection. Now you need to add elements to it. When you add an element, two important steps happen in the background:

1. The information is vectorized (as configured in the collection definition)
2. The HNSW index is updated to optimize search (as you saw in the lectures). This occurs in the backend and you don't see it, but this can make the process take a bit of time

Adding elements is completed using a `collection.batch`, which adds additional useful features. For example, it will let you decide how many objects to send in each batch, handle errors during import, and improve performance by reducing the number of individual network calls. In this example, one element is added at a time, with only a single concurrent request at a time.

You can add a uuid (unique identifier id) to each element you add, so this prevents duplicate entries in your database. 

Let's see in practice!

In [ ]:
# Set up a batch process with specified fixed size and concurrency
with collection.batch.fixed_size(batch_size=1, concurrent_requests=1) as batch:
    # Iterate over a subset of the dataset
    for document in tqdm(data): # tqdm is a library to show progress bars
            # Generate a UUID based on the article_content text for unique identification
            uuid = generate_uuid5(document)

            # Add the object to the batch with properties and UUID. 
            # properties expects a dictionary with the keys being the properties.
            batch.add_object(
                properties=document,
                uuid=uuid,
            )

Awesome! Now you have a collection with vectors! You can check the number of vectors using `len(collection)`:

In [ ]:
len(collection)

<a id='3'></a>
## 3 - Querying on a collection

In this section, you will learn how to query on a collection. You can:

- Query on metadata
- Query with semantic search
- Query with BM25
- Query with filtering

Let's see some examples.

<a id='3-1'></a>
### 3.1 Filters

Before diving into querying, let's understand the Filters. Filters are a way of restricting your search on some criteria. They are very flexible. You usually pass them as an argument in a query. Let's have an example to illustrate it.

In [ ]:
# Here we are fetching 2 objects with a filter by property, filtering by 'user_ratings, only objects with value greater or equal to 3.5'
result = collection.query.fetch_objects(limit = 2, filters = Filter.by_property('user_ratings').greater_or_equal(3.5))

The result is an object called QueryReturn:

In [ ]:
result

You can access its objects by `result.objects`

In [ ]:
result.objects

So, each element in the list is an element of the collection. 

In [ ]:
obj = result.objects[0]

You can check their properties, which is a dictionary.

In [ ]:
obj.properties

In this course, the way of filtering is `.by_property`. You will see more Filtering examples as other query methods are explained.

<a id='3-2'></a>
### 3.2 Semantic Search

You can use semantic search to query over your collection. This uses the vectors to compute distances between them and return the closest ones. You must pass a query, which will be vectorized and then compared over the elements on your collection. The method is `.near_text`.

In [ ]:
result = collection.query.near_text(query = 'I want suggestions to travel during Winter. I want cheap places.', limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print(obj.properties)
    print()

You can also already query over the elements with `budget = Low`:

In [ ]:
result = collection.query.near_text(query = 'I want suggestions to travel during Winter. I want cheap places.', 
                                    filters = Filter.by_property('budget').equal('Low'),
                                    limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print(obj.properties)
    print()

You can also pass a list on the possible values on a filter, by using `.contains_any`:

In [ ]:
result = collection.query.near_text(query = 'I want suggestions to travel during Winter. I want cheap places.', 
                                    filters = Filter.by_property('budget').contains_any(['Low','Moderate']),
                                    limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print(obj.properties)
    print()

<a id='3-3'></a>
### 3.3 BM25 search

To perform BM25 search, just run `colections.query.bm25`, the usual parameters `query`, `limit` and `filters` can be passed.

In [ ]:
result = collection.query.bm25(query = 'I want suggestions to travel during Winter. I want cheap places.', 
                                    filters = Filter.by_property('budget').contains_any(['Low','Moderate']),
                                    limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print(obj.properties)
    print()

<a id='3-4'></a>
### 3.4 Hybrid Search

This search is the RRF search you saw in the lectures. Apart from the standard parameters for querying, you can pass an `alpha` to control how much of BM25 you want in to mix in.

In [ ]:
result = collection.query.hybrid(query = 'I want suggestions to travel during Winter. I want cheap places.', 
                                    filters = Filter.by_property('budget').contains_any(['Low','Moderate']),
                                    alpha = 0.3,
                                    limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print(obj.properties)
    print()

<a id='3-5'></a>
### 3.5 Reranking

You can easily perform reranking with Weaviate by passing a new argument to a search. Let's try with semantic search!

**Remark**:
The reranker does not work. Since I was not able to connect a reranker when creating the collection

In [ ]:
# Don't forget to close the client!
client.close()